In [6]:
# imports
import time
import pandas as pd
import requests
from datetime import datetime
from riotwatcher import LolWatcher, ApiError
from pathlib import Path

# repository root: prefer file-based parent when available, else two levels up
try:
    ROOT = Path(__file__).resolve().parents[1]
except NameError:
    ROOT = Path().resolve().parents[1]


In [ ]:
import time
import json
import requests
import pandas as pd
from datetime import datetime
from pathlib import Path
from riotwatcher import LolWatcher, ApiError

try:
    ROOT = Path(__file__).resolve().parents[1]
except NameError:
    ROOT = Path().resolve().parents[1]

REGION = "americas"
PLATFORM = "na1"
OUT_DIR = ROOT / "data" / "match_data"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── API key ────────────────────────────────────────────────────────────────────
def load_key():
    for p in [
        Path.cwd() / "riot_api_key.txt",
        Path.cwd() / "src" / "riot_api_key.txt",
        ROOT / "riot_api_key.txt",
    ]:
        if p.exists():
            return p.read_text().strip()
    raise FileNotFoundError("riot_api_key.txt not found. Place it in the repo root or src/.")

api_key = load_key()
watcher = LolWatcher(api_key)
print(f"Loaded API key: {api_key[:10]}... (length: {len(api_key)})")

# ── Parsing ────────────────────────────────────────────────────────────────────
def parse_match(match_json):
    """Parse a full Riot Match-v5 JSON into one row per participant."""
    meta = match_json.get("metadata", {})
    info = match_json.get("info", {})
    match_id = meta.get("matchId") or match_json.get("gameId")

    # Team-level objectives
    team_map = {}
    for t in info.get("teams", []):
        tid = t.get("teamId")
        obj = t.get("objectives", {})
        team_map[tid] = {
            "team_win":               t.get("win", False),
            "team_baron_kills":       obj.get("baron",       {}).get("kills", 0),
            "team_dragon_kills":      obj.get("dragon",      {}).get("kills", 0),
            "team_tower_kills":       obj.get("tower",       {}).get("kills", 0),
            "team_inhibitor_kills":   obj.get("inhibitor",   {}).get("kills", 0),
            "team_rift_herald_kills": obj.get("riftHerald",  {}).get("kills", 0),
        }

    rows = []
    for p in info.get("participants", []):
        tid = p.get("teamId")
        team = team_map.get(tid, {})
        kills   = p.get("kills",   0)
        deaths  = p.get("deaths",  0)
        assists = p.get("assists", 0)

        rows.append({
            # Identity
            "match_id":        match_id,
            "puuid":           p.get("puuid"),
            "summonerName":    p.get("riotIdGameName") or p.get("summonerName"),
            "championName":    p.get("championName"),
            "championId":      p.get("championId"),
            "teamId":          tid,
            "teamPosition":    p.get("teamPosition"),
            "role":            p.get("role"),

            # Outcome
            "win":             p.get("win"),

            # KDA
            "kills":           kills,
            "deaths":          deaths,
            "assists":         assists,
            "kda":             (kills + assists) / max(1, deaths),

            # Damage
            "totalDamageToChampions":    p.get("totalDamageDealtToChampions", 0),
            "physicalDamageToChampions": p.get("physicalDamageDealtToChampions", 0),
            "magicDamageToChampions":    p.get("magicDamageDealtToChampions", 0),
            "totalDamageTaken":          p.get("totalDamageTaken", 0),
            "damageSelfMitigated":       p.get("damageSelfMitigated", 0),
            "damageToBuildings":         p.get("damageDealtToBuildings", 0),
            "damageToObjectives":        p.get("damageDealtToObjectives", 0),

            # Economy
            "goldEarned":              p.get("goldEarned", 0),
            "goldSpent":               p.get("goldSpent", 0),

            # CS
            "totalMinionsKilled":      p.get("totalMinionsKilled", 0),
            "neutralMinionsKilled":    p.get("neutralMinionsKilled", 0),
            "cs":                      p.get("totalMinionsKilled", 0) + p.get("neutralMinionsKilled", 0),

            # Vision
            "visionScore":             p.get("visionScore", 0),
            "wardsPlaced":             p.get("wardsPlaced", 0),
            "wardsKilled":             p.get("wardsKilled", 0),
            "visionWardsBought":       p.get("visionWardsBoughtInGame", 0),

            # Crowd control
            "timeCCingOthers":         p.get("timeCCingOthers", 0),
            "totalTimeCCDealt":        p.get("totalTimeCCDealt", 0),

            # Misc performance
            "champLevel":              p.get("champLevel"),
            "doubleKills":             p.get("doubleKills", 0),
            "tripleKills":             p.get("tripleKills", 0),
            "quadraKills":             p.get("quadraKills", 0),
            "pentaKills":              p.get("pentaKills", 0),
            "turretKills":             p.get("turretKills", 0),
            "firstBloodKill":          p.get("firstBloodKill", False),
            "firstBloodAssist":        p.get("firstBloodAssist", False),
            "longestTimeSpentLiving":  p.get("longestTimeSpentLiving"),
            "totalTimeSpentDead":      p.get("totalTimeSpentDead"),
            "spell1Casts":             p.get("spell1Casts", 0),
            "spell2Casts":             p.get("spell2Casts", 0),
            "spell3Casts":             p.get("spell3Casts", 0),
            "spell4Casts":             p.get("spell4Casts", 0),

            # Team objectives
            "team_win":                team.get("team_win"),
            "team_baron_kills":        team.get("team_baron_kills", 0),
            "team_dragon_kills":       team.get("team_dragon_kills", 0),
            "team_tower_kills":        team.get("team_tower_kills", 0),
            "team_inhibitor_kills":    team.get("team_inhibitor_kills", 0),
            "team_rift_herald_kills":  team.get("team_rift_herald_kills", 0),
        })
    return rows

# ── Fetch helpers ──────────────────────────────────────────────────────────────
def fetch_match(region, mid, total_timeout=120):
    start = time.time()
    while time.time() - start < total_timeout:
        try:
            return watcher.match.by_id(region, mid)
        except ApiError as e:
            code = e.response.status_code
            if code == 401:
                raise ValueError("API key expired. Update riot_api_key.txt and restart.")
            elif code == 429:
                wait = int(e.response.headers.get("Retry-After", 2))
                print(f"  Rate limit — waiting {wait}s...")
                time.sleep(wait + 1)
            else:
                print(f"  API error {code} for {mid}, retrying...")
                time.sleep(1)
        except Exception as e:
            print(f"  Error fetching {mid}: {e}")
            time.sleep(1)
    print(f"  Timed out on {mid}, skipping.")
    return None

def get_match_ids(puuid, region, max_matches=None):
    ids, start = [], 0
    while True:
        try:
            batch = watcher.match.matchlist_by_puuid(region, puuid, start=start, count=100)
        except ApiError as e:
            if e.response.status_code == 429:
                wait = int(e.response.headers.get("Retry-After", 2))
                time.sleep(wait + 1)
                continue
            break
        if not batch:
            break
        ids.extend(batch)
        if max_matches and len(ids) >= max_matches:
            ids = ids[:max_matches]
            break
        start += 100
        time.sleep(0.4)
    return ids

# ── Main collection ────────────────────────────────────────────────────────────
def collect(region=REGION, platform=PLATFORM, max_players=None, max_matches_per_player=None,
            out_file="matches_latest.csv", save_every=1000, print_every=10):
    out_path = OUT_DIR / out_file
    print(f"Output: {out_path}")

    # Load challenger entries
    r = requests.get(
        f"https://{platform}.api.riotgames.com/lol/league/v4/challengerleagues/by-queue/RANKED_SOLO_5x5",
        headers={"X-Riot-Token": api_key},
    )
    r.raise_for_status()
    entries = r.json().get("entries", [])
    print(f"Fetched {len(entries)} Challenger players")
    if max_players:
        entries = entries[:max_players]

    all_rows, seen = [], set()

    for i, entry in enumerate(entries, 1):
        # Resolve PUUID from summonerId if needed
        puuid = entry.get("puuid")
        if not puuid:
            try:
                summ = watcher.summoner.by_id(platform, entry["summonerId"])
                puuid = summ["puuid"]
                time.sleep(0.4)
            except Exception as e:
                print(f"Player {i}: couldn't resolve puuid — {e}")
                continue

        print(f"\nPlayer {i}/{len(entries)}: {puuid[:12]}...")
        match_ids = get_match_ids(puuid, region, max_matches=max_matches_per_player)
        print(f"  {len(match_ids)} match IDs")

        for j, mid in enumerate(match_ids, 1):
            if mid in seen:
                continue
            data = fetch_match(region, mid)
            if not data:
                continue
            try:
                rows = parse_match(data)
                all_rows.extend(rows)
                seen.add(mid)
            except Exception as e:
                print(f"  Parse error {mid}: {e}")
            time.sleep(0.4)

            total = len(seen)
            if total % print_every == 0:
                print(f"  [{total} matches] player {i}, match {j}/{len(match_ids)}: {mid}")
            if total % save_every == 0:
                pd.DataFrame(all_rows).to_csv(out_path, index=False)
                print(f"  ── saved {total} matches ──")

        time.sleep(1)

    df = pd.DataFrame(all_rows)
    df.to_csv(out_path, index=False)
    print(f"\nDone. {len(seen)} matches, {len(df)} participant rows → {out_path}")
    return df

# ── Run ────────────────────────────────────────────────────────────────────────
out_name = f"matches_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.csv"
df = collect(max_players=5, out_file=out_name)
print(df.head())


Loaded API key: RGAPI-2673... (length: 42)
Output: C:\Users\Laser\data\match_data\matches_2026-04-12_13-51-31.csv
Fetched 300 Challenger players

Player 1/5: 8TlqDqvmnkxl...
  902 match IDs
  [10 matches] player 1, match 10/902: NA1_5538351905
  [20 matches] player 1, match 20/902: NA1_5537459164
  [30 matches] player 1, match 30/902: NA1_5536898264
  [40 matches] player 1, match 40/902: NA1_5535600772
  [50 matches] player 1, match 50/902: NA1_5533378850
  [60 matches] player 1, match 60/902: NA1_5532281226
  [70 matches] player 1, match 70/902: NA1_5531054972
  [80 matches] player 1, match 80/902: NA1_5529716478
  [90 matches] player 1, match 90/902: NA1_5528710027
  [100 matches] player 1, match 100/902: NA1_5527566822
  [110 matches] player 1, match 110/902: NA1_5524385913
  [120 matches] player 1, match 120/902: NA1_5523345064
  [130 matches] player 1, match 130/902: NA1_5520224335
  [140 matches] player 1, match 140/902: NA1_5518051161
  [150 matches] player 1, match 150/902: NA1

In [ ]:
# Test the API key with a simple request
# Run this cell to verify your Riot API key works before running the main pipeline.

import requests

try:
    api_key = load_key()
    print(f"Testing key: {api_key[:10]}... (length: {len(api_key)})")
    headers = {"X-Riot-Token": api_key}
    # Test with a known summoner (change 'Doublelift' to any valid name if needed)
    test_url = "https://na1.api.riotgames.com/lol/summoner/v4/summoners/by-name/Doublelift"
    r = requests.get(test_url, headers=headers)
    print(f"Response status: {r.status_code}")
    if r.status_code == 200:
        data = r.json()
        print(f"Success! Summoner: {data.get('name')}, Level: {data.get('summonerLevel')}")
    else:
        print(f"Failed: {r.text}")
except Exception as e:
    print(f"Error testing key: {e}")

Testing key: RGAPI-186e... (length: 42)
Response status: 401
Failed: {"status":{"message":"Unknown apikey","status_code":401}}
